# Data Cleaning & Preprocessing


### Objective

This notebook prepares the breast cancer dataset for machine learning.

The preprocessing workflow includes:

- Removing non-predictive and empty columns
- Validating missing values and duplicates
- Encoding the diagnosis target
- Separating features and target
- Performing a stratified train/test split
- Standardizing numerical features
- Preventing data leakage by fitting preprocessing only on training data
- Preparing clean datasets for SVM modelling

In [1]:
### Import Necessary Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

%matplotlib inline
warnings.filterwarnings("ignore")

In [2]:
### Load the dataset
df = pd.read_csv("D:/STUDY/Data_Science_Courses/PROJECTS/2.CancerGuard-SVM/CancerGuard---SVM-Cancer-Detection-API/data/Breast-Cancer_data.csv")
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [3]:
print("Dataset shape:", df.shape)

Dataset shape: (569, 33)


### Step 2 — Remove non-predictive columns
- id → identifier only
- Unnamed: 32 → completely empty CSV artifact

#### 2.1 Check before dropping

In [4]:
columns_to_drop = ["id", "Unnamed: 32"]

for column in columns_to_drop:
    print(
        f"{column}:",
        "Present" if column in df.columns else "Not Present"
    )

id: Present
Unnamed: 32: Present


#### 2.2 Drop the columns

In [5]:
df_clean = df.drop(
    columns=["id", "Unnamed: 32"]
).copy()

print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)

Original shape: (569, 33)
Cleaned shape: (569, 31)


### Step 3 — Validate the cleaned dataset

#### 3.1 Missing values

In [6]:
total_missing = df_clean.isnull().sum().sum()

print("Total missing values:", total_missing)

Total missing values: 0


#### 3.2 Duplicate rows

In [7]:
duplicate_count = df_clean.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


#### 3.3 Verify remaining columns

In [8]:
print("Rows:", df_clean.shape[0])
print("Columns:", df_clean.shape[1])

print("\nTarget column:")
print(df_clean["diagnosis"].value_counts())

print("\nData types:")
print(df_clean.dtypes.value_counts())

Rows: 569
Columns: 31

Target column:
diagnosis
B    357
M    212
Name: count, dtype: int64

Data types:
float64    30
str         1
Name: count, dtype: int64


### Step 4 — Encode the Target Variable

#### 4.1 Encode diagnosis

In [9]:
diagnosis_mapping = {
    "B": 0,
    "M": 1
}

df_clean["diagnosis"] = (
    df_clean["diagnosis"]
    .map(diagnosis_mapping)
)

print("Target variable encoded successfully.")

Target variable encoded successfully.


#### 4.2 Validate the encoding

In [10]:
print("Unique target values:")
print(df_clean["diagnosis"].unique())

print("\nEncoded target distribution:")
print(df_clean["diagnosis"].value_counts().sort_index())

print("\nTarget data type:")
print(df_clean["diagnosis"].dtype)

Unique target values:
[1 0]

Encoded target distribution:
diagnosis
0    357
1    212
Name: count, dtype: int64

Target data type:
int64


#### 4.3 Verify no NaNs were introduced

In [11]:
target_missing = df_clean["diagnosis"].isnull().sum()

print("Missing target values after encoding:", target_missing)

assert target_missing == 0, (
    "Unexpected diagnosis values produced missing targets."
)

assert set(df_clean["diagnosis"].unique()) == {0, 1}, (
    "Target encoding contains unexpected values."
)

print("Target encoding validation passed.")

Missing target values after encoding: 0
Target encoding validation passed.


### Step 5 — Separate Features and Target

- X = predictors
- y = target

#### 5.1 Create X and y

In [12]:
X = df_clean.drop(columns=["diagnosis"])
y = df_clean["diagnosis"]

print("Features (X) shape:", X.shape)
print("Target (y) shape:", y.shape)

Features (X) shape: (569, 30)
Target (y) shape: (569,)


#### 5.2 Validate predictors

In [13]:
print("Number of features:", X.shape[1])

print("\nFeature data types:")
print(X.dtypes.value_counts())

print("\nMissing values in X:", X.isnull().sum().sum())
print("Missing values in y:", y.isnull().sum())

Number of features: 30

Feature data types:
float64    30
Name: count, dtype: int64

Missing values in X: 0
Missing values in y: 0


### Step 6 — Stratified Train/Test Split

#### 6.1 Perform the split

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train/test split completed.")

print("\nX_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

Train/test split completed.

X_train: (455, 30)
X_test : (114, 30)
y_train: (455,)
y_test : (114,)


#### 6.2 Verify stratification

In [16]:
def class_distribution(y_data):
    return pd.DataFrame({
        "Count": y_data.value_counts().sort_index(),
        "Percentage": (
            y_data.value_counts(normalize=True)
            .sort_index() * 100
        ).round(2)
    })


print("Original Dataset:")
display(class_distribution(y))

print("Training Set:")
display(class_distribution(y_train))

print("Test Set:")
display(class_distribution(y_test))

Original Dataset:


,Count,Percentage
diagnosis,,
0,357,62.74
1,212,37.26


Training Set:


,Count,Percentage
diagnosis,,
0,285,62.64
1,170,37.36


Test Set:


,Count,Percentage
diagnosis,,
0,72,63.16
1,42,36.84


### Step 7 — Skewness & Outlier Treatment Strategy

#### 7.1 Recalculate skewness using training data only

In [17]:
train_skewness = X_train.skew().sort_values(
    key=abs,
    ascending=False
)

train_skewness_df = pd.DataFrame({
    "Skewness": train_skewness
})

train_skewness_df["Absolute_Skewness"] = (
    train_skewness_df["Skewness"].abs()
)

display(train_skewness_df)

,Skewness,Absolute_Skewness
area_se,5.497105,5.497105
concavity_se,5.391084,5.391084
fractal_dimension_se,4.085245,4.085245
perimeter_se,3.516777,3.516777
radius_se,3.181121,3.181121
smoothness_se,2.459372,2.459372
symmetry_se,2.245343,2.245343
area_worst,1.902745,1.902745
texture_se,1.809194,1.809194
compactness_se,1.738815,1.738815


#### 7.2 Identify highly skewed features

In [18]:
highly_skewed_features = train_skewness_df[
    train_skewness_df["Absolute_Skewness"] > 1
].index.tolist()

print(
    "Number of highly skewed training features:",
    len(highly_skewed_features)
)

print("\nHighly skewed features:")

for feature in highly_skewed_features:
    print(
        f"{feature:<30} "
        f"{train_skewness_df.loc[feature, 'Skewness']:.3f}"
    )

Number of highly skewed training features: 23

Highly skewed features:
area_se                        5.497
concavity_se                   5.391
fractal_dimension_se           4.085
perimeter_se                   3.517
radius_se                      3.181
smoothness_se                  2.459
symmetry_se                    2.245
area_worst                     1.903
texture_se                     1.809
compactness_se                 1.739
fractal_dimension_worst        1.739
area_mean                      1.720
concave points_se              1.617
symmetry_worst                 1.503
compactness_worst              1.469
concavity_mean                 1.460
compactness_mean               1.271
fractal_dimension_mean         1.259
concave points_mean            1.204
concavity_worst                1.198
perimeter_worst                1.141
radius_worst                   1.116
perimeter_mean                 1.037


### Skewness Treatment Decision

Training-set analysis identified 23 features with absolute skewness
greater than 1.

However, SVM does not require predictor variables to follow a normal
distribution, and the observed skewness may contain diagnostically
relevant information associated with malignant observations.

Therefore, no automatic log or power transformation will be applied
during the primary preprocessing workflow.

The original feature distributions will be retained and standardized.
Alternative transformations may be evaluated later through
cross-validation if they improve model performance.

#### Step 7.3 — Outlier assessment on training data

In [19]:
train_outlier_summary = []

for feature in X_train.columns:

    Q1 = X_train[feature].quantile(0.25)
    Q3 = X_train[feature].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_mask = (
        (X_train[feature] < lower_bound) |
        (X_train[feature] > upper_bound)
    )

    outlier_count = outlier_mask.sum()

    train_outlier_summary.append({
        "Feature": feature,
        "Outlier_Count": outlier_count,
        "Outlier_Percentage":
            (outlier_count / len(X_train)) * 100
    })

train_outlier_df = pd.DataFrame(
    train_outlier_summary
).sort_values(
    "Outlier_Count",
    ascending=False
)

display(train_outlier_df)

,Feature,Outlier_Count,Outlier_Percentage
13,area_se,53,11.648352
12,perimeter_se,34,7.472527
10,radius_se,31,6.813187
15,compactness_se,24,5.274725
16,concavity_se,24,5.274725
19,fractal_dimension_se,23,5.054945
18,symmetry_se,22,4.835165
23,area_worst,21,4.615385
14,smoothness_se,20,4.395604
28,symmetry_worst,20,4.395604


#### Step 7.4 — Check whether outliers disproportionately belong to malignant cases

In [20]:
top_outlier_feature = train_outlier_df.iloc[0]["Feature"]

Q1 = X_train[top_outlier_feature].quantile(0.25)
Q3 = X_train[top_outlier_feature].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outlier_mask = (
    (X_train[top_outlier_feature] < lower_bound) |
    (X_train[top_outlier_feature] > upper_bound)
)

outlier_target_distribution = (
    y_train.loc[X_train.index[outlier_mask]]
    .value_counts()
    .sort_index()
)

print("Feature investigated:", top_outlier_feature)

print("\nIQR bounds:")
print("Lower:", round(lower_bound, 4))
print("Upper:", round(upper_bound, 4))

print("\nTarget distribution among its outliers:")
print(outlier_target_distribution)

Feature investigated: area_se

IQR bounds:
Lower: -23.4475
Upper: 86.6925

Target distribution among its outliers:
diagnosis
1    53
Name: count, dtype: int64


### Outlier Treatment Decision

IQR analysis identified statistical outliers across many predictive
features. However, investigation of `area_se`, the feature containing
the largest number of IQR-defined outliers, showed that all 53 flagged
training observations belonged to the malignant class.

This demonstrates that extreme measurements can contain meaningful
class-related information rather than representing erroneous data.

Therefore:

- IQR-defined observations will not be removed.
- No blanket winsorization or clipping will be applied.
- Extreme values will be retained as potentially informative tumour
  measurements.
- Feature scaling will be fitted using training data only.

### Step 8 — StandardScaler with Leakage Prevention

#### 8.1 Fit StandardScaler on training data ONLY

In [22]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scaler.fit(X_train)

print("StandardScaler fitted successfully.")
print("Number of features learned:", scaler.n_features_in_)

StandardScaler fitted successfully.
Number of features learned: 30


#### Step 8.2 — Transform train and test sets

In [23]:
X_train_scaled_array = scaler.transform(X_train)
X_test_scaled_array = scaler.transform(X_test)

print("Training data transformed:", X_train_scaled_array.shape)
print("Test data transformed:", X_test_scaled_array.shape)

Training data transformed: (455, 30)
Test data transformed: (114, 30)


#### Step 8.3 — Convert back to DataFrames

In [24]:
X_train_scaled = pd.DataFrame(
    X_train_scaled_array,
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled_array,
    columns=X_test.columns,
    index=X_test.index
)

print("X_train_scaled:", X_train_scaled.shape)
print("X_test_scaled :", X_test_scaled.shape)

display(X_train_scaled.head())

X_train_scaled: (455, 30)
X_test_scaled : (114, 30)


,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
10,0.518559,0.891826,0.424632,0.383925,-0.974744,-0.689772,-0.688586,-0.398175,-1.039155,-0.825056,...,0.579798,1.313242,0.466908,0.445983,-0.596155,-0.634722,-0.610227,-0.235744,0.054566,0.021837
170,-0.516364,-1.639710,-0.541349,-0.542961,0.476219,-0.631834,-0.604281,-0.303075,0.521543,-0.454523,...,-0.582459,-1.690291,-0.611934,-0.587014,0.273582,-0.814844,-0.712666,-0.323208,-0.137576,-0.904402
407,-0.368118,0.455515,-0.388250,-0.402970,-1.432979,-0.383927,-0.342175,-0.765459,-0.850857,-0.226171,...,-0.398622,0.181977,-0.475431,-0.420778,-1.622785,-0.391399,-0.431313,-0.890825,-0.675893,-0.144016
430,0.205285,0.726168,0.400330,0.070612,0.243253,2.203585,2.256094,1.213233,0.818474,0.899791,...,-0.000309,0.274191,0.513776,-0.099482,0.418538,2.865970,2.958619,1.977064,-0.075646,1.728848
27,1.243005,0.194195,1.210377,1.206652,-0.111442,0.051348,0.732962,0.713767,-0.427187,-0.822184,...,1.012835,0.223144,0.938517,0.880910,0.073201,-0.277006,0.327775,0.501859,-0.909322,-0.546249


#### Step 8.4 — Verify scaling

In [25]:
scaling_validation = pd.DataFrame({
    "Mean": X_train_scaled.mean(),
    "Std_Dev": X_train_scaled.std(ddof=0)
})

display(scaling_validation)

,Mean,Std_Dev
radius_mean,-1.737316e-16,1.0
texture_mean,3.904081e-16,1.0
perimeter_mean,4.704418e-16,1.0
area_mean,-1.171224e-16,1.0
smoothness_mean,7.242070e-16,1.0
compactness_mean,-5.075305e-17,1.0
concavity_mean,-4.489693e-17,1.0
concave points_mean,2.928061e-17,1.0
symmetry_mean,2.342449e-17,1.0
fractal_dimension_mean,3.669836e-16,1.0


#### Step 8.5 — Check the test set 

In [26]:
test_scaling_check = pd.DataFrame({
    "Mean": X_test_scaled.mean(),
    "Std_Dev": X_test_scaled.std(ddof=0)
})

display(test_scaling_check)

,Mean,Std_Dev
radius_mean,-0.054148,0.920587
texture_mean,-0.149115,1.004300
perimeter_mean,-0.049899,0.912521
area_mean,-0.065009,0.875186
smoothness_mean,0.128268,0.905061
compactness_mean,0.046911,0.893983
concavity_mean,-0.023545,0.873424
concave points_mean,-0.012018,0.884490
symmetry_mean,-0.060598,0.956986
fractal_dimension_mean,0.059351,1.062459


### Step 9 — Final Preprocessing Validation

#### 9.1 Check NaN and infinite values

In [27]:
print("NaN values:")
print("X_train_scaled:", X_train_scaled.isna().sum().sum())
print("X_test_scaled :", X_test_scaled.isna().sum().sum())
print("y_train       :", y_train.isna().sum())
print("y_test        :", y_test.isna().sum())

print("\nInfinite values:")
print(
    "X_train_scaled:",
    np.isinf(X_train_scaled.to_numpy()).sum()
)
print(
    "X_test_scaled :",
    np.isinf(X_test_scaled.to_numpy()).sum()
)

NaN values:
X_train_scaled: 0
X_test_scaled : 0
y_train       : 0
y_test        : 0

Infinite values:
X_train_scaled: 0
X_test_scaled : 0


#### 9.2 Verify feature consistency

In [28]:
assert list(X_train_scaled.columns) == list(X_test_scaled.columns)
assert X_train_scaled.shape[1] == 30
assert X_test_scaled.shape[1] == 30

print("Training and test feature structures are identical.")
print("Number of features:", X_train_scaled.shape[1])

Training and test feature structures are identical.
Number of features: 30


#### 9.3 Verify index alignment

In [29]:
assert X_train_scaled.index.equals(y_train.index)
assert X_test_scaled.index.equals(y_test.index)

print("Training feature/target indices aligned:", True)
print("Test feature/target indices aligned:", True)

Training feature/target indices aligned: True
Test feature/target indices aligned: True


#### 9.4 Verify train/test separation

In [30]:
overlapping_indices = set(X_train_scaled.index).intersection(
    set(X_test_scaled.index)
)

print(
    "Number of overlapping train/test rows:",
    len(overlapping_indices)
)

assert len(overlapping_indices) == 0

print("Train/test separation validation passed.")

Number of overlapping train/test rows: 0
Train/test separation validation passed.




#### Cleaning

- The raw dataset contained 569 observations and 33 columns.
- `id` was removed because it is a non-predictive identifier.
- `Unnamed: 32` was removed because it contained 100% missing values.
- No duplicate observations were identified.
- No missing values remained in the predictive features.

#### Target Encoding

The diagnosis target was encoded as:

- Benign (`B`) = **0**
- Malignant (`M`) = **1**

The original target distribution was preserved:

- Benign: 357 (62.74%)
- Malignant: 212 (37.26%)

#### Train/Test Split

A stratified 80/20 train-test split was performed using a fixed
random state.

- Training set: 455 observations
- Test set: 114 observations
- Predictive features: 30

Stratification preserved approximately the same class distribution
in the training and test datasets.

#### Skewness

Training-set analysis identified 23 features with absolute skewness
greater than 1.

No automatic log or power transformation was applied because SVM
does not require normally distributed predictors and extreme
measurements may contain diagnostically relevant information.

Alternative transformations may be evaluated later using
cross-validation if required.

#### Outliers

IQR-based analysis identified statistical outliers across many
features.

For `area_se`, 53 training observations were identified as IQR
outliers, and all 53 belonged to the malignant class.

Therefore, IQR-defined outliers were retained because removing them
could eliminate clinically meaningful malignant observations.

No blanket outlier removal, clipping, or winsorization was applied.

#### Feature Scaling

StandardScaler was used because SVM is sensitive to differences in
feature scale.

To prevent data leakage:

1. The train/test split was performed before scaling.
2. StandardScaler was fitted only on `X_train`.
3. The fitted scaler transformed `X_train`.
4. The same fitted scaler transformed `X_test`.

Training features have approximately zero mean and unit standard
deviation. Test-set features are transformed exclusively using
statistics learned from the training data.

#### Final Prepared Data

- `X_train_scaled`: (455, 30)
- `X_test_scaled`: (114, 30)
- `y_train`: (455,)
- `y_test`: (114,)